# Explorar relatório de download

Abre o relatório CSV mais recente gerado pelo download (CLI ou pelo app web) — eles ficam na pasta `relatorios/` com timestamp no nome — e ajuda a conferir os matches, ver falhas e filtrar por status.

**Dica:** selecione a `.venv` do projeto como kernel (canto superior direito) para ter todas as dependências (pandas, matplotlib). Rode as células de cima para baixo (Shift+Enter).

In [ ]:
from pathlib import Path
import pandas as pd

# Pasta onde ficam os relatórios (padrão: relatorios/ dentro do projeto)
REPORTS_DIR = Path('relatorios')

# Pega o relatório mais recente (por data de modificação)
reports = sorted(REPORTS_DIR.glob('download_report*.csv'), key=lambda p: p.stat().st_mtime)
assert reports, f'Nenhum download_report*.csv encontrado em {REPORTS_DIR.resolve()}'

report_path = reports[-1]   # mais recente. Troque por reports[0], reports[-2]… p/ outro
print('Relatórios encontrados:')
for r in reports:
    print('  ', r.name)
print('\nUsando:', report_path.name)

df = pd.read_csv(report_path)
print('Linhas:', len(df))
df.head()

## Resumo por status

In [ ]:
resumo = df['status'].value_counts()
print(resumo.to_string())
print('\nTotal:', len(df))

# Gráfico é opcional: se matplotlib não estiver instalado, segue sem quebrar.
try:
    resumo.plot(kind='bar', title='Faixas por status', rot=0)
except Exception as e:
    print('\n(gráfico indisponível — instale matplotlib ou use a .venv como kernel:', type(e).__name__ + ')')

## Falhas (rode o download de novo p/ tentar só elas — o resume pula as prontas)

In [ ]:
falhas = df[df['status'] == 'failed']
if falhas.empty:
    print('Nenhuma falha. 🎉')
else:
    for _, r in falhas.iterrows():
        print(f"#{r['n']:>3}  {r['artist']} - {r['title']}")
        print(f"      motivo: {r['message']}\n")
falhas[['n', 'artist', 'title', 'message']]

## Conferir os matches escolhidos

Útil para checar se algum link saiu errado (cover, versão estendida, etc.). A coluna `chosen_title` é o título do vídeo escolhido no YouTube.

In [ ]:
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)
df[['n', 'artist', 'title', 'chosen_title', 'status', 'chosen_url']]

## Buscar uma faixa específica

In [ ]:
termo = 'guns'   # <- digite parte do artista ou título
mask = df['artist'].str.contains(termo, case=False, na=False) | df['title'].str.contains(termo, case=False, na=False)
df[mask][['n', 'artist', 'title', 'chosen_title', 'status', 'chosen_url', 'message']]

## Comparar vários relatórios (ex: playlists/execuções diferentes)

In [ ]:
for r in reports:
    d = pd.read_csv(r)
    ok = (d['status'] == 'ok').sum()
    sk = (d['status'] == 'skipped').sum()
    fail = (d['status'] == 'failed').sum()
    print(f"{r.name:40}  total={len(d):3}  ok={ok:3}  skip={sk:3}  falhas={fail:3}")